# 05. 문서함 Agent


| 주어지는 것 | 직접 만들 것 |
|-------------|-------------|
| `pdf_samples/*.pdf` 7개 | RAG 함수 + Tool 3개 + Agent |
| `_catalog/index.json` + `*.summary.txt` | `get_document_catalog()` **읽기만** |



---
## 0. 문서함 (주어진 데이터)

**PDF 경로:** `samples/pdf_samples/*.pdf` 만 사용

| 파일 | 유형 |
|------|------|
| `학칙.pdf` | 규정 |
| `보도자료.pdf` | 보도자료 (SK하이닉스 2026 1분기 실적) |
| `A survey on large language model based autonomous agents.pdf` | 논문 |
| `data2vec.pdf` | 논문 |
| `Physics-Guided Deep Learning.pdf` | 논문 |
| `supervised-contrastive-learning-Paper.pdf` | 논문 |
| `NeurIPS-2023-analyzing-vision-transformers-...-Paper-Conference.pdf` | 논문 |

**카탈로그 :** `pdf_samples/_catalog/`
- `index.json` — pdf_name, category, keywords, summary_file
- `학칙.summary.txt` 등 — 문서별 한 줄 요약

---
## 1. 전체 그림

```
[주어짐] _catalog/          [만들 것] Tool 함수
         index.json    →    get_document_catalog()  ← 읽기만
         *.summary.txt

[주어짐] *.pdf         →    build_pdf_index() + search_in_document()

                              ↓
                         run_agent()  ← AGENT_TOOLS + 루프
```

---
## 2. Part A — RAG 기초 함수 3개 (03 노트북)


필요한 Libary 다운

In [1]:
# !pip install openai python-dotenv pymupdf

import json
import re
from pathlib import Path

import pymupdf
from dotenv import load_dotenv
from openai import OpenAI

BASE = Path('.').resolve()
load_dotenv(BASE.parent / '.env')

client = OpenAI()
DOC_LIBRARY = BASE / 'samples' / 'pdf_samples'
CATALOG_DIR = DOC_LIBRARY / '_catalog'
INDEX_CACHE = {}  # pdf_name → chunks

Chunk 나누는 함수

In [4]:
def split_text(text, chunk_size=280, overlap=60):
    text = re.sub(r'\s+', ' ', text.strip()) #공백지우기 전처리
    if len(text) <= chunk_size:
        return [text]
    chunks, start = [], 0 #chunks, start 정의
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end]) #start부터 chunks size까지 하나로 묶음
        if end >= len(text):
            break
        start = max(0, end - overlap) #chunk size에서 overlap 만큼 뺴줌 -> overlap은 겹치는 부분
    return chunks

def tokenize_agent(text):
    return {t.lower() for t in re.findall(r'[가-힣a-zA-Z0-9]+', text) if len(t) > 1} #가나다 순으로 띄어쓰기 기준 토큰화

def search_agent(query, top_k=3):
    q = tokenize_agent(query)
    scored = []
    for item in chunks: 
        score = len(q & tokenize_agent(item['text']))
        if score > 0:
            scored.append((score, item))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [{'score': s, **it} for s, it in scored[:top_k]]

1. chunk 생성하는 함수
2. 질문을 토큰화 하는 함수
3. 토큰화된 질문과 chunk간 top_k 기준으로 score화

---
## 3. Part B — PDF 인덱스 함수 2개

In [ ]:
import pymupdf

def read_pdf_text(pdf_path: Path) -> str:
    doc = pymupdf.open(pdf_path)
    return "\n".join(page.get_text() for page in doc)

def chunk_stem(pdf_name: str) -> str:
    stem = Path(pdf_name).stem
    return re.sub(r"[^\w가-힣\-]+", "_", stem)[:40].strip("_")

def build_pdf_index(pdf_name: str) -> list[dict]:
    if pdf_name in INDEX_CACHE:
        return INDEX_CACHE[pdf_name]
    pdf_path = DOC_LIBRARY / pdf_name #pdf 경로 DOC_LIBARY :  BASE / 'samples' / 'pdf_samples'
    if not pdf_path.exists(): #없으면 빈 list 반환
        return []
    text = read_pdf_text(pdf_path)
    prefix = chunk_stem(pdf_name)
    index = [
        {
            "chunk_id": f"{prefix}_C{i:03d}",
            "source_pdf": pdf_name,
            "text": chunk,
        }
        for i, chunk in enumerate(split_text(text), 1)
    ]
    INDEX_CACHE[pdf_name] = index
    return index

---
## 4. Part C — Tool 함수 3개 (OpenAI에 등록할 것)

| 함수 | 필수 | 설명 |
|------|------|------|
| `list_documents()` | ✅ | pdf_samples PDF 파일명 JSON |
| `get_document_catalog()` | ✅ | `_catalog/index.json` + summary txt **읽기** |
| `search_in_document(pdf_name, query, top_k)` | ✅ | build_pdf_index + search_chunks → JSON |

In [ ]:
def list_documents()

---
## 5. Part D — Agent 합치기

1. `AGENT_TOOLS` — 위 3개 함수 스키마
2. `TOOL_FUNCTIONS` — 이름 → 함수 dict
3. `AGENT_SYSTEM` — catalog 먼저, 그다음 search
4. `run_agent(question)` — **tool_calls 없을 때까지 루프**

---
## 6. 데모 — 6개 통과

| # | 질문 | 기대 PDF |
|---|------|----------|
| 1 | 문서함에 어떤 PDF가 있어? | catalog |
| 2 | 석사학위과정 수업연한은? | `학칙.pdf` |
| 3 | LLM autonomous agent 서베이 주제는? | survey 논문 |
| 4 | data2vec modality 세 가지는? | `data2vec.pdf` |
| 5 | 2026년 1분기 SK하이닉스 영업이익은? | `보도자료.pdf` |
| 6 | ViT class embedding 논문 한 줄로 | NeurIPS ViT 논문 |

In [ ]:
DEMO = [
    '문서함에 어떤 PDF가 있어?',
    '석사학위과정 수업연한은?',
    'LLM autonomous agent 서베이 논문의 주제는?',
    'data2vec가 다루는 modality 세 가지는?',
    '2026년 1분기 SK하이닉스 영업이익은?',
    'Vision Transformer class embedding 논문이 뭐 다루는지 한 줄로',
]

for q in DEMO:
    print('=' * 60, '\nQ:', q, '\n', run_agent(q))